In [1]:
import pandas as pd
from pandasql import sqldf

## Подготовка

Исходные данные обезличены (конфиденциальная информация удалена) и приведены к формату .csv. Дубликаты строк в таблицах не удалялись, так как отражают реальную кратность событий: один клиент может иметь несколько заказов в день, один заказ — несколько идентичных товаров и т.д.

In [2]:
# Укажите путь к данным на Вашем ПК, пример пути:
your_path = 'C:/Users/Admin/Desktop'

orders = pd.read_csv(f'{your_path}/dashboard-pvz-demo-main/data/demo_orders.csv')
drivers = pd.read_csv(f'{your_path}/dashboard-pvz-demo-main/data/demo_drivers.csv')

In [3]:
total_orders = ('''
SELECT CASE WHEN moment >= '2026-02-01' AND moment < '2026-03-01' THEN 'Февраль'
            WHEN moment >= '2026-03-01' AND moment < '2026-04-01' THEN 'Март'
            WHEN moment >= '2026-04-01' AND moment < '2026-05-01' THEN 'Апрель' END AS period_name, 
            DATE(moment) AS event_date, moment, orderNumber, postingId, isReturned, returnReason
FROM orders
''')
total_orders = sqldf(total_orders)

In [4]:
total_drivers = ('''
SELECT CASE WHEN receivingDate >= '2026-02-01' AND receivingDate < '2026-03-01' THEN 'Февраль'
            WHEN receivingDate >= '2026-03-01' AND receivingDate < '2026-04-01' THEN 'Март'
            WHEN receivingDate >= '2026-04-01' AND receivingDate < '2026-05-01' THEN 'Апрель' END AS period_name, *
FROM drivers
''')
total_drivers = sqldf(total_drivers)

# Анализ

### Среднее число клиентов в час												

In [5]:
orders_clients = ('''
WITH hours_active AS (
        SELECT period_name, event_date, CAST(strftime('%H', moment) AS INTEGER) AS event_hour, 
-- orderNumber = '12345-0001' → id клиента = '12345' (удаляю последние 5 символов, включая дефис и номер заказа)
               substr(orderNumber, 1, length(orderNumber) - 5) AS order_number
        FROM total_orders),
    clients AS (          
        SELECT period_name, event_date, event_hour, COUNT(DISTINCT order_number) AS clients
        FROM (
-- Вписываю нестандартное время выдачи в рабочие часы, т.к. это редкие аномалии:
            SELECT CASE WHEN event_hour = 8 THEN 9
                        WHEN event_hour = 21 THEN 20
                        ELSE event_hour 
                   END event_hour, period_name, event_date, order_number
            FROM hours_active) t
        GROUP BY period_name, event_date, event_hour)

SELECT period_name, event_hour, AVG(clients) AS avg_clients
FROM clients
GROUP BY period_name, event_hour
''')
orders_clients = sqldf(orders_clients)

# вариант округления на python
orders_clients['avg_clients'] = orders_clients['avg_clients'].round(1)

pt_orders_clients = orders_clients.pivot_table(
    values='avg_clients', 
    index='period_name', 
    columns='event_hour'
)

pt_orders_clients = pt_orders_clients.reindex(['Февраль', 'Март', 'Апрель'])

### Данные по дням

In [6]:
total_day = ('''
WITH orders AS (
        SELECT period_name, event_date, substr(orderNumber, 1, length(orderNumber) - 5) AS order_number, isReturned
        FROM total_orders),
    count_items_days AS (
        SELECT period_name, event_date, 
            SUM(CASE WHEN isReturned = 0 THEN 1 ELSE 0 END) AS count_item_giveout,
            SUM(CASE WHEN isReturned = 1 THEN 1 ELSE 0 END) AS count_item_cancel,
            COUNT(DISTINCT order_number) AS clients
        FROM orders
        GROUP BY period_name, event_date),
    acceptance_volume AS (
        SELECT period_name, DATE(receivingDate) AS event_date,
            SUM(COALESCE(receivedPostingQtyTotal, 0)) AS items_get_fact,
            SUM(COALESCE(shortageQty, 0)) AS items_shortage,
            SUM(COALESCE(excessQty, 0)) AS items_excess
        FROM total_drivers
        GROUP BY period_name, DATE(receivingDate))
    
SELECT cid.*,
       COALESCE(av.items_get_fact, 0) AS items_get_fact,
       COALESCE(av.items_shortage, 0) AS items_shortage,
       COALESCE(av.items_excess, 0) AS items_excess
FROM count_items_days cid
LEFT JOIN acceptance_volume av 
ON cid.event_date = av.event_date 
AND cid.period_name = av.period_name
ORDER BY CASE cid.period_name
              WHEN 'Февраль' THEN 0
              WHEN 'Март' THEN 1
              WHEN 'Апрель' THEN 2
         END, cid.event_date
''')
total_day = sqldf(total_day)

### Клиенты по числу товаров в заказе		

In [7]:
orders_per_client = ('''
WITH orders AS (
            SELECT period_name, event_date, substr(orderNumber, 1, length(orderNumber) - 5) AS id_client
            FROM total_orders),
    item_per_client AS (
            SELECT period_name, event_date, id_client, COUNT(id_client) AS count_items
            FROM orders
            GROUP BY period_name, event_date, id_client)

SELECT period_name,
       CASE WHEN count_items BETWEEN 5 AND 9 THEN '5-9'
            WHEN count_items BETWEEN 10 AND 14 THEN '10-14'
            WHEN count_items >= 15 THEN '15+'
-- Группы '5-9', '10-14' и '15+' — текст, поэтому и числовые группы (1-4) тоже перевожу в текст для единого типа данных:
            ELSE CAST(count_items AS TEXT)
        END AS items_in_order, COUNT(*) AS orders
FROM item_per_client
GROUP BY period_name, items_in_order
ORDER BY CASE period_name
              WHEN 'Февраль' THEN 1
              WHEN 'Март' THEN 2
              WHEN 'Апрель' THEN 3
          END,
          CASE items_in_order
               WHEN '5-9' THEN 997
               WHEN '10-14' THEN 998
               WHEN '15+' THEN 999
          END
''')
orders_per_client = sqldf(orders_per_client)

### Причины отказа

In [8]:
reason_item_cancel = ('''
WITH all_causes AS (
-- Предварительно создаю список всех причин для корректного LEFT JOIN реальных данных, важны причины даже с нулями:
        SELECT 'Проблемы с качеством товара (брак товара)' AS reason UNION ALL
        SELECT 'Привезли другой товар' UNION ALL
        SELECT 'Нет части товаров из комплекта/набора' UNION ALL
        SELECT 'Изменил решение о покупке/Товар не подошёл' UNION ALL
        SELECT 'Отказ клиента: поврежден товар' UNION ALL
        SELECT 'Отказ клиента при передаче' UNION ALL
        SELECT 'Существенно повреждена упаковка'),
    period_reason AS (
        SELECT DISTINCT t.period_name, a.reason
        FROM all_causes a
        JOIN total_orders t)

SELECT p.period_name, p.reason, COUNT(q.reason) AS count_items
FROM period_reason p
LEFT JOIN (
    SELECT period_name, returnReason AS reason
    FROM total_orders
    WHERE isReturned = 1) q
ON p.period_name = q.period_name
AND p.reason = q.reason
GROUP BY p.period_name, p.reason
ORDER BY CASE p.period_name
              WHEN 'Февраль' THEN 1
              WHEN 'Март' THEN 2
              WHEN 'Апрель' THEN 3
         END,
         CASE p.reason
              WHEN 'Проблемы с качеством товара (брак товара)' THEN 993
              WHEN 'Привезли другой товар' THEN 994
              WHEN 'Нет части товаров из комплекта/набора' THEN 995
              WHEN 'Изменил решение о покупке/Товар не подошёл' THEN 996
              WHEN 'Отказ клиента: поврежден товар' THEN 997
              WHEN 'Отказ клиента при передаче' THEN 998
              WHEN 'Существенно повреждена упаковка' THEN 999
         END
''')
reason_item_cancel = sqldf(reason_item_cancel)

### Нагрузка клиентов по дням недели

In [9]:
weekly_clients = ('''
WITH clients AS (
            SELECT period_name, event_date, substr(orderNumber, 1, length(orderNumber) - 5) AS id_client, 
                   CAST(strftime('%H', moment) AS INTEGER) AS event_hour,
                   CASE strftime('%w', event_date)
                        WHEN '0' THEN 'Вс'
                        WHEN '1' THEN 'Пн'
                        WHEN '2' THEN 'Вт'
                        WHEN '3' THEN 'Ср'
                        WHEN '4' THEN 'Чт'
                        WHEN '5' THEN 'Пт'
                        WHEN '6' THEN 'Сб'
                    END AS day_name
            FROM total_orders),
    hourly AS (
-- Один клиент может зайти за заказом несколько раз в день (до и после приемки). Считаю его в каждый час, но в рамках часа - уникально:
            SELECT period_name, event_date, day_name, event_hour, COUNT(DISTINCT id_client) AS clients_per_hour
            FROM clients
            GROUP BY period_name, event_date, day_name, event_hour),
    daily AS (
            SELECT period_name, event_date, day_name, SUM(clients_per_hour) AS clients_per_day
            FROM hourly
            GROUP BY period_name, event_date, day_name)

SELECT period_name, day_name, ROUND(AVG(clients_per_day), 1) AS avg_clients
FROM daily
GROUP BY period_name, day_name
ORDER BY CASE period_name
              WHEN 'Февраль' THEN 1
              WHEN 'Март' THEN 2
              WHEN 'Апрель' THEN 3
         END,
         CASE day_name
            WHEN 'Пн' THEN 0
            WHEN 'Вт' THEN 1
            WHEN 'Ср' THEN 2
            WHEN 'Чт' THEN 3
            WHEN 'Пт' THEN 4
            WHEN 'Сб' THEN 5
            WHEN 'Вс' THEN 6
        END
''')
weekly_clients = sqldf(weekly_clients)

pt_weekly_clients = weekly_clients.pivot_table(
    values='avg_clients', 
    index='period_name', 
    columns='day_name'
)

pt_weekly_clients = pt_weekly_clients[['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']]
pt_weekly_clients = pt_weekly_clients.reindex(['Февраль', 'Март', 'Апрель'])

### Час привоза товаров

In [10]:
driver_hour = ('''
WITH drivers AS (
        SELECT period_name, DATE(receivingDate) AS event_date, CAST(strftime('%H', receivingDate) AS INTEGER) AS event_hour, receivingDate
        FROM total_drivers),
    working_hours AS (
-- Время привоза не зависит от времени работы пункта:
        SELECT 6 AS event_hour UNION SELECT 7 UNION SELECT 8 UNION 
        SELECT 9 UNION SELECT 10 UNION SELECT 11 UNION SELECT 12 UNION SELECT 13 UNION 
        SELECT 14 UNION SELECT 15 UNION SELECT 16 UNION SELECT 17 UNION SELECT 18 UNION 
        SELECT 19 UNION SELECT 20 UNION SELECT 21 UNION SELECT 22),
    period_hour AS (
        SELECT DISTINCT t.period_name, w.event_hour
        FROM working_hours w
        JOIN total_drivers t)

SELECT p.period_name, p.event_hour, COUNT(d.receivingDate) AS count_drivers
FROM period_hour p
LEFT JOIN drivers d
ON p.period_name = d.period_name
AND p.event_hour = d.event_hour
GROUP BY p.period_name, p.event_hour
ORDER BY CASE p.period_name
              WHEN 'Февраль' THEN 1
              WHEN 'Март' THEN 2
              WHEN 'Апрель' THEN 3
         END
''')
driver_hour = sqldf(driver_hour)

pt_driver_hour = driver_hour.pivot_table(
    values='count_drivers', 
    index='period_name', 
    columns='event_hour'
)

pt_driver_hour = pt_driver_hour.astype(int)

### Общие данные в периоде

In [11]:
total_period = ('''
WITH unique_clients AS (
        SELECT period_name, COUNT(DISTINCT substr(orderNumber, 1, length(orderNumber) - 5)) AS unique_clients
        FROM total_orders
        GROUP BY period_name),
    acceptance_time AS (
        SELECT period_name, ROUND(12 - AVG(acceptance_time), 1) AS other_time, 
               ROUND(AVG(acceptance_time), 1) AS avg_accept_time,
               ROUND(AVG(acceptance_time) / 12 * 100, 1) AS perc_accept_time
        FROM (
            SELECT period_name, DATE(receivingDate) AS event_date, (julianday(closingDate) - julianday(receivingDate)) * 24 AS acceptance_time
            FROM total_drivers) t
        GROUP BY period_name),
    total_day_sum AS (
        SELECT period_name,
               SUM(items_get_fact + items_excess) AS total_accepted, 
               SUM(count_item_giveout) AS total_item_get, 
               SUM(count_item_cancel) AS total_item_return, 
               SUM(clients) AS total_orders, 
               ROUND(SUM(count_item_giveout + count_item_cancel) * 1.0 / SUM(clients), 1) AS items_per_order
        FROM total_day
        GROUP BY period_name)

SELECT td.period_name, td.total_accepted, td.total_item_get, td.total_item_return, 
       at.other_time, at.avg_accept_time, at.perc_accept_time,
       nc.unique_clients, td.total_orders,
       td.items_per_order
FROM total_day_sum td
LEFT JOIN acceptance_time at
ON td.period_name = at.period_name
LEFT JOIN unique_clients nc
ON td.period_name = nc.period_name
ORDER BY CASE td.period_name
              WHEN 'Февраль' THEN 1
              WHEN 'Март' THEN 2
              WHEN 'Апрель' THEN 3
         END
''')
total_period = sqldf(total_period)

# Сохранение в таблицу

In [12]:
with pd.ExcelWriter(f'{your_path}/dashboard-pvz-demo-main/processed/total_pvz_demo.xlsx') as writer:    
    total_day.to_excel(writer, sheet_name='Данные по дням', index=False)
    total_period.to_excel(writer, sheet_name='Общие данные в периоде', index=False)
    pt_driver_hour.to_excel(writer, sheet_name='Час привоза товаров', index=True)
    pt_orders_clients.to_excel(writer, sheet_name='Клиенты в час ср', index=True)
    pt_weekly_clients.to_excel(writer, sheet_name='Клиенты по дням недели', index=True)
    reason_item_cancel.to_excel(writer, sheet_name='Причины отказов', index=False)
    orders_per_client.to_excel(writer, sheet_name='Товары в заказе', index=False)